# Building a believable company

**Question.** Can a synthetic ledger feel like ordinary company life while carrying six different kinds of fraud? We start with an honest 1,000-claim company, alter one suspect at a time, and keep the untouched baseline for comparison. Red marks identify known injected rows; a large value alone is never a conviction.

In [ ]:
import sys
sys.path.insert(0, "../backend")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chisquare, chi2_contingency, binomtest, ttest_1samp
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import IsolationForest
from auditor.generator.base import (CATEGORY_PROFILES, HONEST_SIGMA, HONEST_Z_CLIP, CEO_MULTIPLIER, make_suspects, make_honest_transactions)
from auditor.generator.cases import build_case, build_honest_case, CASE_START
from auditor.generator.schemes import inject, SCHEME_CATCHERS
from auditor.generator.red_herrings import add_herring
from auditor.tools.registry import run_tool
sns.set_theme(style="darkgrid", context="notebook", palette=["#F5A524", "#4CC9F0", "#E5484D", "#3DD68C"])
plt.rcParams.update({"figure.facecolor":"#0E1117", "axes.facecolor":"#161B24", "axes.labelcolor":"#E8E6E1", "text.color":"#E8E6E1", "xtick.color":"#E8E6E1", "ytick.color":"#E8E6E1", "grid.color":"#394252", "savefig.facecolor":"#0E1117"})
AMBER, CYAN, RED = "#F5A524", "#4CC9F0", "#E5484D"

In [ ]:
rng = np.random.default_rng(17)
suspects = make_suspects(rng, include_ceo=True)
honest = make_honest_transactions(rng, suspects, n_rows=1000, start=CASE_START)
fraudster = next(s.id for s in suspects if s.role != "CEO")
people = pd.DataFrame([{"id":s.id,"name":s.name,"role":s.role} for s in suspects])
print(people)
print(f"{len(honest)} honest claims; demonstration suspect: {fraudster}")

## 1. The shape of honest money

**Question.** Why do amounts look skewed in rupees but approximately bell shaped after a log transform? Each category uses `median × exp(0.40 × clipped Normal(0,1))`; the 2.5σ policy clip trims extreme honest claims. The nominal category medians span roughly 2.5 decades. Their log centres are deliberately staggered to cover many leading digit positions, though the six gaps are **not exactly** 1/6 decade in the current constants. The combined mixture is therefore only approximately Benford.

In [ ]:
profile = pd.DataFrame(CATEGORY_PROFILES, index=["median_INR","weight"]).T
profile["log10_median"] = np.log10(profile["median_INR"])
profile["gap_decades"] = profile["log10_median"].diff()
print(profile.round(3))
plot = honest.assign(log10_amount=np.log10(honest.amount))
g = sns.displot(data=plot, x="log10_amount", col="category", col_wrap=3, bins=25, color=CYAN, height=2.4, facet_kws={"sharex":True,"sharey":False})
g.set_axis_labels("log₁₀(amount in INR)", "Claims")
g.fig.suptitle("Honest expenses: bell-shaped within each category on a log scale", y=1.03)
plt.show()

**Takeaway.** Category-specific baselines make more sense than a single rupee cutoff. The spread of category centres also makes the first digit uneven in the aggregate.

In [ ]:
digits = np.floor(honest.amount / 10**np.floor(np.log10(honest.amount))).astype(int)
observed = digits.value_counts(normalize=True).reindex(range(1,10), fill_value=0)
expected = pd.Series(np.log10(1+1/np.arange(1,10)), index=range(1,10))
fig, ax=plt.subplots(figsize=(8,3.5))
x=np.arange(1,10); ax.bar(x-.18, observed, width=.36, label="Observed honest ledger", color=CYAN); ax.bar(x+.18, expected, width=.36, label="Benford: log₁₀(1+1/d)", color=AMBER)
ax.set(xlabel="First significant digit d",ylabel="Share of claims",title="A mixture across scales approaches Benford's law",xticks=x); ax.legend(); plt.show()
print(f"Maximum absolute digit-share gap: {(observed-expected).abs().max():.3f}")

## 2. Five suspects, ordinary habits

**Question.** Does a favourite vendor imply a ghost vendor? Each suspect has two preferred vendors per category and chooses among them 75% of the time. Shared vendors still appear. The heatmap shows counts, while the grouped table makes the habit visible.

In [ ]:
vendor_counts=pd.crosstab(honest.employee_id,honest.vendor)
print(honest.groupby(["employee_id","vendor"]).size().rename("claims").reset_index().sort_values(["employee_id","claims"],ascending=[True,False]).groupby("employee_id").head(3))
fig,ax=plt.subplots(figsize=(12,3.6)); sns.heatmap(vendor_counts, cmap="YlOrBr", ax=ax, cbar_kws={"label":"Claims"}); ax.set(title="Vendor-use habits by suspect",xlabel="Vendor",ylabel="Employee"); plt.xticks(rotation=70,ha="right"); plt.tight_layout(); plt.show()

**Takeaway.** A person visiting a vendor more often is normal. The ghost-vendor scheme creates a vendor used by nobody else, repeatedly.

## 3. Six ways to alter the same ledger

**Question.** What did each injector actually change? Every call starts from `honest`, with a fresh seeded RNG. IDs returned by `inject()` identify known fraud rows. The figures use the before/after values or added rows, rather than assuming every anomaly is fraud.

In [ ]:
schemes=["big_invoice","duplicates","fabricated","weekend","inflated_ratio","under_limit"]
injected={s:inject(s,honest,fraudster,np.random.default_rng(100+i)) for i,s in enumerate(schemes)}
assert len(honest)==1000
for s,(frame,ids) in injected.items():
    print(f"{s}: {len(ids)} known fraud rows; ledger rows {len(honest)} → {len(frame)}")

In [ ]:
frame,ids=injected["big_invoice"]
before=honest.set_index("txn_id").loc[ids,"amount"]; after=frame.set_index("txn_id").loc[ids,"amount"]
fig,ax=plt.subplots(figsize=(7,3.2)); pos=np.arange(len(ids)); ax.bar(pos-.2,before,.4,label="Before",color=CYAN); ax.bar(pos+.2,after,.4,label="After",color=RED); ax.set(xticks=pos,xticklabels=ids,ylabel="Claim amount (INR)",title="Big invoice: existing claims multiplied far above their category norm"); ax.legend(); plt.show()

**Takeaway.** The invoice changes in place; the fair comparison is the employee’s own category.

In [ ]:
frame,ids=injected["duplicates"]
copies=frame[frame.txn_id.isin(ids)].copy(); originals=honest.merge(copies[["vendor","amount","employee_id"]],on=["vendor","amount","employee_id"])
fig,ax=plt.subplots(figsize=(9,3.3)); ax.scatter(pd.to_datetime(originals.date),originals.amount,color=CYAN,label="Original"); ax.scatter(pd.to_datetime(copies.date),copies.amount,color=RED,marker="D",label="Resubmitted copy")
for _,copy in copies.iterrows():
    match=originals[(originals.vendor==copy.vendor)&(originals.amount==copy.amount)].iloc[0]
    ax.plot(pd.to_datetime([match.date,copy.date]),[copy.amount]*2,color=AMBER,alpha=.7)
ax.set(xlabel="Date",ylabel="Amount (INR)",title="Duplicates: same vendor and amount reappear days later with new invoice IDs"); ax.legend(); plt.tight_layout(); plt.show()

**Takeaway.** A new invoice ID hides a repeated payment; the pair is revealed by employee, vendor, amount, and date.

In [ ]:
frame,ids=injected["fabricated"]
def digit_share(values):
    d=np.floor(values/10**np.floor(np.log10(values))).astype(int)
    return d.value_counts(normalize=True).reindex(range(1,10),fill_value=0)
old=digit_share(honest.loc[honest.employee_id==fraudster,"amount"]); new=digit_share(frame.loc[frame.employee_id==fraudster,"amount"])
fig,ax=plt.subplots(figsize=(8,3.2)); ax.plot(range(1,10),old,marker="o",label="Before: honest",color=CYAN); ax.plot(range(1,10),new,marker="o",label="After: fabricated",color=RED); ax.plot(range(1,10),expected,ls="--",label="Benford",color=AMBER); ax.set(xlabel="First significant digit",ylabel="Share of claims",title="Fabricated: the suspect's digit fingerprint changes",xticks=range(1,10)); ax.legend(); plt.show()

**Takeaway.** The forger replaces all their amounts and adjusts deal revenue to preserve each old margin.

In [ ]:
frame,ids=injected["weekend"]
before=honest.assign(weekend=pd.to_datetime(honest.date).dt.day_of_week>=5).groupby("employee_id").weekend.sum(); after=frame.assign(weekend=pd.to_datetime(frame.date).dt.day_of_week>=5).groupby("employee_id").weekend.sum()
fig,ax=plt.subplots(figsize=(7,3.2)); ax.bar(before.index,before,color=CYAN,label="Before"); ax.bar(after.index,after-before,bottom=before,color=RED,label="Added weekend claims"); ax.set(xlabel="Employee",ylabel="Weekend claims",title="Weekend scheme: extra Saturday and Sunday spending"); ax.legend(); plt.show()

**Takeaway.** The signal is an unusually large weekend share for one person, not a one-week company-wide spike.

In [ ]:
frame,ids=injected["inflated_ratio"]
old=(honest.set_index("txn_id").loc[ids,"amount"]/honest.set_index("txn_id").loc[ids,"revenue"]); new=(frame.set_index("txn_id").loc[ids,"amount"]/frame.set_index("txn_id").loc[ids,"revenue"])
fig,ax=plt.subplots(figsize=(7,3.2)); ax.scatter(old,new,alpha=.55,color=RED); ax.plot([0,.15],[0,.15],ls="--",color=AMBER,label="No padding"); ax.set(xlabel="Original expense / deal revenue",ylabel="Padded expense / deal revenue",title="Inflated ratio: travel and dining claims rise while revenue stays fixed"); ax.legend(); plt.show()

**Takeaway.** Padding a whole class of claims shifts the suspect above the normal expense–revenue relationship.

In [ ]:
frame,ids=injected["under_limit"]
added=frame[frame.txn_id.isin(ids)]; original=honest[honest.employee_id==fraudster]
fig,ax=plt.subplots(figsize=(8,3.3)); ax.scatter(original.hour,original.amount,alpha=.35,color=CYAN,label="Original claims"); ax.scatter(added.hour,added.amount,s=65,color=RED,label="Ghost vendor claims"); ax.axhline(50000,color=AMBER,ls="--",label="INR 50,000 approval limit"); ax.set(xlabel="Hour of day",ylabel="Amount (INR)",yscale="log",title="Under limit: overnight payments to one exclusive vendor"); ax.legend(); plt.show()
print("Ghost vendor:",added.vendor.unique().tolist())

**Takeaway.** The joint pattern matters: repeated exclusive vendor, 1–4 AM, and amounts just under approval.

## 4. Innocent red herrings

**Question.** What else could look suspicious? The CEO’s fourfold multiplier is built into honest generation. The bonus is added to **every** employee in one working week. Neither is an injected fraud row.

In [ ]:
ceo=next(s.id for s in suspects if s.role=="CEO")
ceo_median=honest.groupby(["employee_id","category"]).amount.median().unstack()
relative=ceo_median.loc[ceo]/ceo_median.drop(ceo).median()
bonus=add_herring("diwali",honest,suspects,np.random.default_rng(44))
bonuses=bonus[bonus.category=="festival_bonus"]
fig,axes=plt.subplots(1,2,figsize=(12,3.6)); relative.plot.bar(ax=axes[0],color=AMBER); axes[0].axhline(CEO_MULTIPLIER,color=CYAN,ls="--"); axes[0].set(title="CEO category medians vs peers",xlabel="Category",ylabel="Median ratio"); axes[0].tick_params(axis="x",rotation=60)
sns.scatterplot(data=bonuses,x="date",y="amount",hue="employee_id",s=85,ax=axes[1]); axes[1].set(title="Diwali bonus: one legitimate payment each",xlabel="Working day",ylabel="Bonus (INR)"); axes[1].tick_params(axis="x",rotation=45); plt.tight_layout(); plt.show()
assert bonuses.employee_id.nunique()==len(suspects) and len(bonuses)==len(suspects)
print("CEO:",ceo,"; bonus dates:",sorted(bonuses.date.unique()))

**Takeaway.** Both effects have an explicit company policy or shared event behind them. Context prevents a dramatic chart from becoming a false accusation.

## 5. Can a case be reproduced?

**Question.** Does the same seed rebuild the same evidence? Determinism makes a classroom result auditable.

In [ ]:
first=build_case(6,seed=42); second=build_case(6,seed=42)
pd.testing.assert_frame_equal(first.df,second.df)
assert first.truth==second.truth
print(f"Identical {len(first.df)}-row frames and identical truth for level 6, seed 42.")

**Takeaway.** The game can replay a case exactly, including the hidden fraud labels; only the visible ledger is handed to the player.